In [0]:
from pyspark.sql.functions import to_date, col
from pyspark.sql.types import DecimalType, TimeType, DateType
import yaml
from delta.tables import DeltaTable

In [0]:
%sql
SELECT * FROM nedbank_ovation.bronze.transactions

#Silver cleaning from bronze tables

#Accounts

In [0]:
with open("../config/pipeline_config.yaml") as pc:
  config = yaml.safe_load(pc)

# accounts
df_acc = spark.read.table("nedbank_ovation.bronze.accounts")
df_acc = df_acc.dropDuplicates(["account_id"])
df_acc = df_acc.withColumn("open_date", to_date(col("open_date"), "yyyy-MM-dd"))
df_acc = df_acc.withColumn("last_activity_date", to_date(col("last_activity_date"), "yyyy-MM-dd"))
df_acc = df_acc.withColumn("credit_limit", col("credit_limit").cast(DecimalType(18,2)))
df_acc = df_acc.withColumn("current_balance", col("current_balance").cast(DecimalType(18,2)))

# customers
df_cust = spark.read.table("nedbank_ovation.bronze.customers")
df_cust = df_cust.dropDuplicates(["customer_id"])
df_cust = df_cust.withColumn("dob", to_date(col("dob"), "yyyy-MM-dd"))

# Transactions
df_trans = spark.read.table("nedbank_ovation.bronze.transactions")
df_trans = df_trans.dropDuplicates(["transaction_id"])
df_trans = df_trans.withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd"))
df_trans = df_trans.withColumn("amount", col("amount").cast(DecimalType(18,2)))
#Location flattening
df_trans = df_trans.withColumn("location_province", col("location").getItem("province"))
df_trans = df_trans.withColumn("location_city", col("location").getItem("city"))
df_trans = df_trans.withColumn("location_coordinates", col("location").getItem("coordinates"))
df_trans = df_trans.withColumn("device_id", col("metadata").getItem("device_id"))
#Metadata flattening
df_trans = df_trans.withColumn("session_id", col("metadata").getItem("session_id"))
df_trans = df_trans.withColumn("retry_flag", col("metadata").getItem("retry_flag").cast("boolean"))
df_trans = df_trans.withColumn("ingestion_time", col("ingestion_time").cast(DateType()))
#Dropping the object columns that are now flattened
df_trans = df_trans.drop("location", "metadata")

data_frames = {
    "accounts": {
        "key" : "account_id",
        "df": df_acc
    }, 
    "customers": {
        "key" : "customer_id",
        "df": df_cust
    }, 
    "transactions": {
        "key" : "transaction_id",
        "df": df_trans
    }
}

#For databricks

for tblname, details in data_frames.items():
    if spark.catalog.tableExists(tblname):
        delta_tbl = DeltaTable.forName(spark, f"nedbank_ovation.silver.{tblname}")
        delta_tbl.alias("t").merge(
            details["df"].alias("s"),
            f"t.{details["key"]} = s.{details["key"]}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        details["df"].write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"nedbank_ovation.silver.{tblname}")

#Writting dataframes to the silver layer
for tblname, details in data_frames.items():
    if DeltaTable.isDeltaTable(spark, f"{config["output"]["silver_path"]}/{tblname}"):
        delta_tbl = DeltaTable.forPath(spark, f"{config["output"]["silver_path"]}/{tblname}")
        delta_tbl.alias("t").merge(
            details["df"].alias("s"),
            f"t.{details["key"]} = s.{details["key"]}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        details["df"].write.format("delta").mode("overwrite").save(f"{config["output"]["silver_path"]}/{tblname}")
        


In [0]:
%sql
SELECT 
  count(*)
FROM nedbank_ovation.silver.transactions